# kaggle-vllm 0.1.1 dual-T4 acceptance test

Run this notebook in a Kaggle GPU session configured with two NVIDIA Tesla T4 GPUs. It is an unexecuted acceptance checklist: committed outputs are intentionally absent, and success must not be claimed until every required cell completes in Kaggle. The optional Qwen section downloads the multi-gigabyte TP=2 persistent checkpoint.

In [ ]:
!nvidia-smi -L

In [ ]:
import platform
import sys

print(sys.executable)
print(sys.version)
print(platform.platform())

In [ ]:
%pip install --upgrade --no-cache-dir kaggle-vllm==0.1.1

In [ ]:
import kaggle_vllm

print(kaggle_vllm.__version__)
assert kaggle_vllm.__version__ == "0.1.1"

In [ ]:
!kaggle-vllm fingerprint

In [ ]:
!kaggle-vllm bootstrap --dry-run --strict

In [ ]:
!kaggle-vllm bootstrap --strict

In [ ]:
from kaggle_vllm import activate_runtime

assert activate_runtime(), "bootstrap manifest/runtime was not found"
!kaggle-vllm doctor

In [ ]:
import torch
import vllm

print("torch", torch.__version__)
print("torch CUDA", torch.version.cuda)
print("vLLM", vllm.__version__)
print("visible GPUs", torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    print(index, props.name, (props.major, props.minor))
assert torch.cuda.device_count() == 2
assert all(
    "Tesla T4" in torch.cuda.get_device_name(index)
    for index in range(torch.cuda.device_count())
)

In [ ]:
from kaggle_vllm import KaggleLLM
from vllm import SamplingParams

smoke_llm = KaggleLLM(
    model="facebook/opt-125m",
    tensor_parallel_size=2,
    max_model_len=512,
    gpu_memory_utilization=0.70,
)
smoke_outputs = smoke_llm.generate(
    ["Kaggle dual-T4 tensor parallelism is"],
    SamplingParams(max_tokens=16, temperature=0.0),
)
print(smoke_outputs[0].outputs[0].text)

## Optional: persistent Qwen TP=2 sharded-state acceptance

This section downloads the large topology-aware checkpoint. It is not a standard Transformers checkpoint and is validated only with `load_format="sharded_state"` and `tensor_parallel_size=2`.

In [ ]:
from huggingface_hub import snapshot_download

qwen_path = snapshot_download("waqasm86/kaggle-vllm-models")
qwen_llm = KaggleLLM(
    model=qwen_path,
    load_format="sharded_state",
    tensor_parallel_size=2,
    dtype="float16",
    enforce_eager=True,
    disable_custom_all_reduce=True,
    max_model_len=2048,
    gpu_memory_utilization=0.70,
)
qwen_outputs = qwen_llm.generate(
    ["Explain tensor parallelism in one sentence."],
    SamplingParams(max_tokens=32, temperature=0.0),
)
print(qwen_outputs[0].outputs[0].text)